In [25]:
from Problem import Problem
from Algorithm import VariableNeighborhoodSearch
from Neighborhood import Swap, Reversion, Insertion, Slide, ETN, RS, SPS, SRPS, FixedCostSwap, TransportCostSwap
from Solution import Solution
import pickle
import copy
import numpy as np

data = Problem()
data.loadFile("../data/data_10.npz")

with open("../solution_test.pickle", "rb") as f:
    solution = pickle.load(f)
solution.evaluate(data)
print("Initial solution cost:", solution.FX)

Initial solution cost: 10257396.618116293


In [26]:
new_solution = copy.deepcopy(solution)

# Helper to swap bits in a 1D binary array
def swap_bits(arr, idx_remove, idx_add):
    lst = list(arr)
    lst[idx_remove], lst[idx_add] = lst[idx_add], lst[idx_remove]
    return np.array(lst)

chromosome_attr, chromosome = 'S1', solution.S1  # Example for Pistachio Factories -> Pistachio Consumers
row_mask = chromosome  # binary vector representing active(1)/inactive(0)

print(f"Initial chromosome: {chromosome}")
print(f" ")

#print(f"Transportation Matrix: {solution.P}")

print("Transportation Matrix:")
matrix_to_print = solution.P.toarray() if hasattr(solution.P, 'toarray') else solution.P

np.set_printoptions(precision=2, suppress=True, linewidth=100)
print(matrix_to_print)
np.set_printoptions()  # Reset to default
print(f" ")

print(f"Demand: {data.Dp}")
print(f" ")

print(f"Transportation Costs:")
print(np.array2string(data.Cw, formatter={'float_kind':lambda x: "%.2f" % x}))


Initial chromosome: [ 3  6 12 15  5 11 17  4  8 16 13  9  1 14 18 19 20  7  2 10]
 
Transportation Matrix:
[[  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.   317.88 288.98 265.62   0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [211.73 204.29 335.88 274.76   0.     0.     0.   303.89 246.63 344.68]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]]
 
Demand: [211.73 204.29 335.88 274.76 317.88 288.98 265.62 303.89 246.63 344.68]
 
Transportation Costs:
[118.73 101.98 137.34 116.91 113.39 109.77 114.70 149.62

In [27]:
# 2) Extrair a matriz de fluxo P para o cromossomo S1
flow_mat = solution.P.toarray()  # shape = (K, J)
K, J = flow_mat.shape

In [28]:
# 3) Listar pares (k,j) de arestas ativas e inativas
#    - Aresta ativa: P[k,j] > 0
#    - Aresta inativa: P[k,j] == 0
ativa_pairs = np.argwhere(flow_mat > 0)
inativa_pairs = np.argwhere(flow_mat == 0)

print("\nArestas ativas (k,j):")
print(ativa_pairs)
print("\nArestas inativas (k,j):")
print(inativa_pairs)


Arestas ativas (k,j):
[[1 4]
 [1 5]
 [1 6]
 [6 0]
 [6 1]
 [6 2]
 [6 3]
 [6 7]
 [6 8]
 [6 9]]

Arestas inativas (k,j):
[[0 0]
 [0 1]
 [0 2]
 [0 3]
 [0 4]
 [0 5]
 [0 6]
 [0 7]
 [0 8]
 [0 9]
 [1 0]
 [1 1]
 [1 2]
 [1 3]
 [1 7]
 [1 8]
 [1 9]
 [2 0]
 [2 1]
 [2 2]
 [2 3]
 [2 4]
 [2 5]
 [2 6]
 [2 7]
 [2 8]
 [2 9]
 [3 0]
 [3 1]
 [3 2]
 [3 3]
 [3 4]
 [3 5]
 [3 6]
 [3 7]
 [3 8]
 [3 9]
 [4 0]
 [4 1]
 [4 2]
 [4 3]
 [4 4]
 [4 5]
 [4 6]
 [4 7]
 [4 8]
 [4 9]
 [5 0]
 [5 1]
 [5 2]
 [5 3]
 [5 4]
 [5 5]
 [5 6]
 [5 7]
 [5 8]
 [5 9]
 [6 4]
 [6 5]
 [6 6]
 [7 0]
 [7 1]
 [7 2]
 [7 3]
 [7 4]
 [7 5]
 [7 6]
 [7 7]
 [7 8]
 [7 9]
 [8 0]
 [8 1]
 [8 2]
 [8 3]
 [8 4]
 [8 5]
 [8 6]
 [8 7]
 [8 8]
 [8 9]
 [9 0]
 [9 1]
 [9 2]
 [9 3]
 [9 4]
 [9 5]
 [9 6]
 [9 7]
 [9 8]
 [9 9]]


Ao invés de trocar as prioridades dos dois pares de arestas, a gente pode:

* Olhar a aresta inativa mais barata; setar o cliente dessa aresta com maior prioridade; diminuir a prioridade do restante no intervalo por -1. Exemplo: se o cliente que tinha prioridade 9 receber 20, então todos com prioridade de 10 a 20 vão receber -1.

In [29]:
if ativa_pairs.size == 0 or inativa_pairs.size == 0:
    print("\nNão há arestas ativas ou inativas para troca. Operador não faz nada.")
else:
    # Imprimir custo inicial para comparação
    print(f"\nCusto inicial: {solution.FX}")
    
    # 4) Calcular custo de cada aresta via data.Cp
    custos_ativas = np.array([data.Cp[k, j] for (k, j) in ativa_pairs])
    custos_inativas = np.array([data.Cp[k, j] for (k, j) in inativa_pairs])

    # 5) Identificar aresta ativa mais cara e inativa mais barata
    idx_exp = np.argmax(custos_ativas)
    k_exp, j_exp = ativa_pairs[idx_exp]
    print(f"\nAresta ativa mais cara: (k={k_exp}, j={j_exp}), custo = {custos_ativas[idx_exp]:.2f}")

    idx_chp = np.argmin(custos_inativas)
    k_chp, j_chp = inativa_pairs[idx_chp]
    print(f"Aresta inativa mais barata: (k={k_chp}, j={j_chp}), custo = {custos_inativas[idx_chp]:.2f}")

    # 6) Ajuste de prioridades só no cliente de j_chp
    chrom_old = solution.S1.copy()
    chrom_new = chrom_old.copy()

    # índices
    K, J = flow_mat.shape

    # prioridade antiga do cliente escolhido
    p_old = chrom_old[K + j_chp]
    # prioridade máxima atual
    p_max = chrom_old.max()

    # 1) Descer em 1 ponto tudo que esteja acima de p_old
    mask = chrom_new > p_old
    chrom_new[mask] -= 1

    # 2) Colocar o cliente desejado no topo (p_max)
    chrom_new[K + j_chp] = p_max

    # 7) Monta nova solução e reavalia
    new_solution = copy.deepcopy(solution)
    new_solution.S1 = chrom_new
    new_cost = new_solution.evaluate(data)

    print("Prioridade antiga do cliente:", p_old)
    print("Prioridade máxima:", p_max)
    print("Cromossomo antes:", chrom_old)
    print("Cromossomo depois:", chrom_new)

    # Agora, atribua e reavalie:
    new_solution = copy.deepcopy(solution)
    new_solution.S1 = chrom_new
    new_cost = new_solution.evaluate(data)
    
    # Comparação final dos custos
    print(f"\nCusto inicial: {solution.FX}")
    print(f"Novo custo: {new_cost}")
    print(f"Diferença: {new_cost - solution.FX:.2f}")
    print(f"Melhoria (%): {((solution.FX - new_cost) / solution.FX * 100):.2f}%")
    
    flow_mat_new = new_solution.P.toarray()
    np.set_printoptions(precision=2, suppress=True, linewidth=100)
    print(flow_mat_new)
    np.set_printoptions()  # reset


Custo inicial: 10257396.618116293

Aresta ativa mais cara: (k=6, j=1), custo = 33.74
Aresta inativa mais barata: (k=7, j=2), custo = 20.04
Prioridade antiga do cliente: 1
Prioridade máxima: 20
Cromossomo antes: [ 3  6 12 15  5 11 17  4  8 16 13  9  1 14 18 19 20  7  2 10]
Cromossomo depois: [ 2  5 11 14  4 10 16  3  7 15 12  8 20 13 17 18 19  6  1  9]

Custo inicial: 10257396.618116293
Novo custo: 10286132.168055719
Diferença: 28735.55
Melhoria (%): -0.28%
[[  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.   317.88 288.98 265.62   0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [211.73 204.29   0.   274.76   0.     0.     0.   303.89 246.63 344.68]
 [  0.     0.     0.     0.


### Teste 2: 
Em vez de olhar arestas isoladas, vamos elevar a prioridade da fonte globalmente mais barata (aquela cujo custo total para atender todos os clientes é mínimo).

In [30]:
# --- 1) calcular fonte globalmente mais barata ---
C1 = data.Cp
row_sums = C1.sum(axis=1)
k_min = np.argmin(row_sums)

# --- 2) ajustar prioridades ---
chrom_old = solution.S1.copy()
chrom_new = chrom_old.copy()
p_old = chrom_old[k_min]
p_max = chrom_old.max()

mask = chrom_new > p_old
chrom_new[mask] -= 1
chrom_new[k_min] = p_max

print("Fonte barata:", k_min, "custo total =", row_sums[k_min])
print("Cromossomo antes:", chrom_old)
print("Cromossomo depois:", chrom_new)

# --- 3) aplicar e reavaliar ---
new_sol = copy.deepcopy(solution)
new_sol.S1 = chrom_new
new_cost = new_sol.evaluate(data)

# Comparação final dos custos
print(f"\nCusto inicial: {solution.FX}")
print(f"Custo após boost: {new_cost}")
print(f"Diferença: {new_cost - solution.FX:.2f}")
print(f"Melhoria (%): {((solution.FX - new_cost) / solution.FX * 100):.2f}%")

# --- 4) mostrar nova P ---
P_new = new_sol.P.toarray()
np.set_printoptions(precision=2, suppress=True)
print("\nNova matriz P (S1):")
print(P_new)
np.set_printoptions()  # volta ao default

Fonte barata: 9 custo total = 269.24040738360566
Cromossomo antes: [ 3  6 12 15  5 11 17  4  8 16 13  9  1 14 18 19 20  7  2 10]
Cromossomo depois: [ 3  6 12 15  5 11 16  4  8 20 13  9  1 14 17 18 19  7  2 10]

Custo inicial: 10257396.618116293
Custo após boost: 10222247.60436108
Diferença: -35149.01
Melhoria (%): 0.34%

Nova matriz P (S1):
[[  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.  ]
